In [2]:
from PIL import ImageDraw, ImageFont, Image
import os
import json
from pathlib import Path
from tqdm.auto import tqdm

from dexlib.tools.kh2_exporter.kh2_exporter import (
    Kh2Record,
    FileImageProvider,
    Kh2Annotation,
    Kh2Attribute,
    Kh2Object,
    Kh2ObjectMetadata,
    Kh2Bbox,
    Kh2Exporter,
)
from toolbox.pdf import read_pages
from toolbox.ocr.textbox import Textbox

In [5]:
from kh2_records_deserializer import Kh2RecordsDeserializer
kh_drecord = Kh2RecordsDeserializer().deserialize_from_dir("./pttep_exported_2_docs_2025-04-03_15-42-54")

FileNotFoundError: [Errno 2] No such file or directory: './pttep_exported_2_docs_2025-04-03_15-42-54/index.json'

In [3]:
def generate_kh2_sample(annotations: list[Textbox], label: str) -> list[Kh2Object]:
    khobj = []
    for ann in annotations:
        x_coords = [ann.box[i][0] for i in range(4)]
        y_coords = [ann.box[i][1] for i in range(4)]
        x1 = min(x_coords)
        x2 = max(x_coords)
        y1 = min(y_coords)
        y2 = max(y_coords)
        khobj.append(
            Kh2Object(
                text=ann.text,
                label=label,
                bbox=Kh2Bbox(x1=x1, y1=y1, x2=x2, y2=y2),
                metadata=Kh2ObjectMetadata()
            )
        )
    return khobj

In [4]:
def is_valid_image(image_source: str | Path | Image.Image) -> bool:
    
    MAX_DIMENSION = 5000
    try:
        # Open image if source is file path
        if isinstance(image_source, (str, Path)):
            image = Image.open(image_source)
        else:
            image = image_source
            
        # Verify image integrity
        image.verify()
        
        # Check dimensions
        width, height = image.size
        if width > MAX_DIMENSION or height > MAX_DIMENSION:
            raise ValueError(
                f"Image dimensions ({width}x{height}) exceed maximum allowed size "
                f"({MAX_DIMENSION}x{MAX_DIMENSION})"
            )
            
        return True
        
    except Exception as e:
        # Log the error if needed (consider using logging module in production)
        print(f"Invalid image: {str(e)}")
        return False

In [7]:
path2img = Path("./output/images")
path2ocr = Path("./output/json")
pdf_files = Path("./pdf")
output = Path("./output")
output.mkdir(exist_ok=True, parents=True)

kh2_records = []
for file in pdf_files.rglob("*.pdf"):
    print(file)
    img_dir = path2img / file.stem
    ocr_dir = path2ocr / file.stem
    num_pages = len(list(img_dir.rglob("*.png")))
    for i in tqdm(range(num_pages)):
        json_file = ocr_dir / f"{i}.json"
        if not json_file.exists():
            print("Err json")
            break
        with json_file.open() as f:
            data = json.load(f)
        list_textboxes = [Textbox.model_validate(tb) for tb in data]
        objects = generate_kh2_sample(list_textboxes, "text")
        if file.stem == "12":
            objects.extend(kh_drecord[1].annotation.objects)
            im = "./pttep_exported_2_docs_2025-04-03_15-42-54/pages/67e3ed8b736e3ae0a8ba207b-p0/image.png"
        elif file.stem == "66":
            objects.extend(kh_drecord[0].annotation.objects)
            im = "./pttep_exported_2_docs_2025-04-03_15-42-54/pages/67e3ed8b736e3ae0a8ba207b-p0/image.png"
        else:
            im = str(img_dir / f"{i}.png")
        
        if not is_valid_image(im):
            raise Exception(f"Invalid image: {im}")
    
        kh2_record = Kh2Record(
            image_provider=FileImageProvider(im),
            annotation=Kh2Annotation(
                fields=[
                    Kh2Attribute(key="File name", value=str(file)),
                ],
                objects=objects,
            ),
            file_path=str(file),
            page_number=i,
            upload_image_key=None,
        )
        kh2_records.append(kh2_record)

In [8]:
zip_path = "kh2_pttep.zip"
Kh2Exporter().export(kh2_records=kh2_records, output_zip_path=zip_path)

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/tmpqjju3f_o/output/index.json'

kh2-upload-example-from-zip

In [31]:
from kh2_upload_example_misc import generate_kh2_sample
from kh_documents_uploader_service import KhUploaderConfig, KhUploaderService
from dexlib.tools.kh2_uploader.kh2_uploader import Kh2Uploader

from datetime import datetime

In [32]:
kh2_uploader_config = KhUploaderConfig(
    kh2_url="http://192.168.92.127:7575",
    kh2_user_email="admin@example.com",
    kh2_user_password="asdfasdf",
    kh2_project_id="67e5556ce25552fc77b1832c",
)

In [33]:
zip_path = zip_path

In [34]:
kh2_uploader = Kh2Uploader(
    api_url=kh2_uploader_config.kh2_url + '/api/v1',
    user_email=kh2_uploader_config.kh2_user_email,
    user_password=kh2_uploader_config.kh2_user_password,
    project_id=kh2_uploader_config.kh2_project_id,
)
run_name = f'run-{datetime.now().strftime("%d/%m/%Y %H%M%S")}'
uploaded_run_id = kh2_uploader.upload_run(zip_path, run_name)
kh2_link = (
    f"{kh2_uploader_config.kh2_url}/user_view/explore_results/{uploaded_run_id}"
)
print(kh2_link)

KeyboardInterrupt: 